In [1]:
from openai import OpenAI
import openai
import os

api_key = os.getenv('OPENAI_API_KEY')
openai.api_key = api_key

In [2]:
%pip install wikipedia

Note: you may need to restart the kernel to use updated packages.


In [18]:
import wikipedia

In [19]:
def wikipedia_search(question):
    wikipedia.set_lang("ko")
    try:
        search_result = wikipedia.search(question)[0]
        wiki_summary = wikipedia.summary(search_result, sentences=5)
    except Exception as e:
        wiki_summary = "위키피디아에서 정보를 찾을 수 없습니다."
    return {"summary": wiki_summary}

In [20]:
wikipedia_search("세종대왕은 누구야?" )

{'summary': '세종(한국 한자: 世宗, 중세 한국어: ·솅조ᇰ, 1397년 5월 15일 (음력 4월 10일) ~ 1450년 3월 30일 (음력 2월 17일))은 조선의 정치가로, 제4대 국왕(재위 : 1418년 9월 9일 ~ 1450년 3월 30일)이다. 태종과 원경왕후의 아들이다. 형인 양녕대군이 폐세자가 되자 세자에 책봉되었으며 태종의 양위를 받아 즉위하였다.\n\n세종은 과학 기술, 예술, 문화, 국방 등 여러 분야에서 다양한 업적을 남겼다. 백성들에게 농사에 관한 책을 펴내었지만 글을 몰라 이해하지 못하는 모습을 보고 누구나 쉽게 배울 수 있는 효율적이고 과학적인 문자 체계인 훈민정음(訓民正音)을 창제하였다.'}

In [ ]:
import json
import wikipedia
from openai import OpenAI


client = OpenAI()


def wikipedia_search(question):
    wikipedia.set_lang("ko")
    try:
        search_result = wikipedia.search(question)[0]
        wiki_summary = wikipedia.summary(search_result, sentences=5)
    except Exception as e:
        wiki_summary = "위키피디아에서 정보를 찾을 수 없습니다."
    return {"summary": wiki_summary}


wikifunc = [{
    "type": "function",
    "function": {
        "name": "wikipedia_search",
        "description": "입력된 질문에 대해 필요하다면 위키피디아에서  정보를 검색합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "question": {
                    "type": "string",
                    "description": " 주제 또는 질문"
                }
            },
            "required": ["question"]
        }
    }
}]

messages = [{"role": "user", "content": "세종대왕에 대해 알려줘"}]

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=messages,
    tools=wikifunc,
    tool_choice="auto"
)


tool_call = response.choices[0].message.tool_calls[0]
args = json.loads(tool_call.function.arguments)
print( args)
result = wikipedia_search(args["question"])
print(result)

# messages = [{"role": "user", "content": "오리 너구리에 대해 알려줄래"},
#             {"role": "function","name": "wikipedia_search",
#              "content":'{summany:위키피디아답변}'}]
messages.append({
    "role": "function",
    "name": "wikipedia_search",
    "content": json.dumps(result)
})

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=messages,
    temperature=0
)


print(response.choices[0].message.content)


{'question': '세종대왕'}
{'summary': '세종(한국 한자: 世宗, 중세 한국어: ·솅조ᇰ, 1397년 5월 15일 (음력 4월 10일) ~ 1450년 3월 30일 (음력 2월 17일))은 조선의 정치가로, 제4대 국왕(재위 : 1418년 9월 9일 ~ 1450년 3월 30일)이다. 태종과 원경왕후의 아들이다. 형인 양녕대군이 폐세자가 되자 세자에 책봉되었으며 태종의 양위를 받아 즉위하였다.\n\n세종은 과학 기술, 예술, 문화, 국방 등 여러 분야에서 다양한 업적을 남겼다. 백성들에게 농사에 관한 책을 펴내었지만 글을 몰라 이해하지 못하는 모습을 보고 누구나 쉽게 배울 수 있는 효율적이고 과학적인 문자 체계인 훈민정음(訓民正音)을 창제하였다.'}
세종대왕(한국어: 世宗, 정치명: 세조 한자: 世宗, 1397년 5월 15일 (음력 4월 10일) ~ 1450년 3월 30일 (음력 2월 17일))은 조선의 제4대 국왕(정치명: 1418년 9월 9일 ~ 1450년 3월 30일)이다. 대왕은 세종과 이성계의 아들들이다. 형식적으로는 조선의 제4대 국왕이지만, 실질적인 통치자는 세종이었다. 

세종대왕은 학문, 과학, 문학, 국방 등 다양한 분야에서 혁신적인 업적을 이루었으며, 백성을 위한 정책과 사회적 변화를 이끌었다. 그의 통치는 조선시대의 전성기를 이루는 중요한 요소 중 하나로 평가되고 있다. 세종대왕은 학문적인 업적과 국방력을 향상시키는 데 큰 관심을 기울였으며, 한글 창제와 세종실록 편찬 등의 사업을 추진했다. 또한 백성을 위한 정책으로 세찬 백성을 위한 법령을 제정하고, 농업 생산성을 향상시키는 농업정책을 시행했다.

세종대왕은 학문적인 업적과 국방력을 향상시키는 데 큰 관심을 기울였으며, 한글 창제와 세종실록 편찬 등의 사업을 추진했다. 또한 백성을 위한 정책으로 세찬 백성을 위한 법령을 제정하고, 농업 생산성을 향상시키는 농업정책을 시행했다. 세종대왕은 조선시대의 위대한 군주로 평가되고 있으며, 그의 업적은 한국 역사상 가장 중요한 것 